In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from tqdm import tqdm
import yaml
from torchgeodemo import autoencoder_train_latent
import torch
import os


#notebook to rest reconstruction error of PCA

random_seed_number = 20210321  # 2021 UK Census date used as random seed
def set_random_seeds(seed):
    # """Set random seeds for reproducibility."""
    torch.manual_seed(seed)  # Set seed for PyTorch


# Set the seed
set_random_seeds(seed=random_seed_number)

def gen_layer_sizes(input_size,latent_size,num_layers,scaling_type = "mul"):

    """
    Generate layer sizes for a neural network based on the input size, latent size, number of layers, and scaling type.

    Args:
        input_size (int): The size of the input layer.
        latent_size (int): The size of the latent layer.
        num_layers (int): The number of layers in the network.
        scaling_type (str): The type of scaling to use. Can be "mul" or "lin".

    Returns:
        list: A list containing the sizes of each layer in the network.
    """
    if scaling_type == "mul":
        # Geometric scaling
        scale_factor = (latent_size / input_size) ** (1 / (num_layers - 1))
        layer_sizes = [int(input_size * scale_factor ** i) for i in range(num_layers)]
    elif scaling_type == "lin":
        # Linear scaling
        step = (latent_size - input_size) / (num_layers - 1)
        layer_sizes = [int(input_size + step * i) for i in range(num_layers)]
    else:
        raise ValueError("Invalid scaling type. Use 'mul' or 'lin'.")

    #for torchgeodemo dont want to include the input layer
    layer_sizes = layer_sizes[1:]  # Exclude input layer size
    return layer_sizes

In [ ]:
#load the cleaned data
data_name = "engcensus_all"
cleaned_data_path = "../data/census_data/engcensus_cleaned_scaled.parquet"
df_scaled = pd.read_parquet(cleaned_data_path)
df_scaled

In [ ]:


epochs = 250
scaling_type = "lin"  # "mul" for geometric scaling, "lin" for linear scaling
# Define the run name and working directory
run_name = f"{epochs}epoch_scan_{scaling_type}"
WORKING_DIR = "../data/AE_outputs/"+data_name+"/"+run_name
os.makedirs(WORKING_DIR, exist_ok=True)
# Output directory for generated YAMLs
OUTPUT_DIR = WORKING_DIR+ "/yamls"
os.makedirs(OUTPUT_DIR, exist_ok=True)

bottleneck_sizes = [2,4,8,16,32,64,100,128]
input_size = len(df_scaled.columns)-1
encoder_layer_sizes = []
for bottleneck_size in bottleneck_sizes:
    encoder_layer_sizes.append(gen_layer_sizes(input_size, bottleneck_size, 4, scaling_type=scaling_type))

def plot_loss(WORKING_DIR,data_nickname,ae_nickname):
    # Load the loss data
    loss_data = pd.read_csv(WORKING_DIR + f"/logs/log_{data_nickname}_{ae_nickname}_v1/version_1/metrics.csv")[['epoch', 'train_loss_epoch', 'recon_loss_epoch']]
    #drop rows with nan
    loss_data = loss_data.dropna()
    plt.plot(loss_data["epoch"], loss_data["train_loss_epoch"], label="Training Loss")
    plt.plot(loss_data["epoch"], loss_data["recon_loss_epoch"], label="Reconstruction Loss")
    plt.show()


In [ ]:
# Set the seed
set_random_seeds(seed=random_seed_number)
# Define the base YAML structure as a dictionary
BASE_YAML = {
    "data": {
        "source": cleaned_data_path,
        "nickname": "PLACEHOLDER",
        "id_col": "OA"
    },
    "working_dir": WORKING_DIR,
    "autoencoder": {
        "nickname": "bottleneck_PLACEHOLDER",
        "version": "1",
        "save_latent": "csv",
        "max_epochs": epochs,
        "batch_size": 0.01,
        "use_covariance_loss": False,
        "encoder": {
            "sizes": "PLACEHOLDER",
            "activation": "LeakyReLU"
        },
        "decoder": {
            "sizes":  "PLACEHOLDER",
            "activation": "LeakyReLU"
        }
    }
}

data_nickname = "census_geodemo"
# Generate YAML files with varying latent space sizes
for encoder_layer_size in encoder_layer_sizes:
    _latent_size = encoder_layer_size[-1]  # Last element is the bottleneck size
    ae_nickname = f"bottleneck_{_latent_size}"
    yaml_config = BASE_YAML.copy()
    yaml_config["autoencoder"]["encoder"]["sizes"] = encoder_layer_size
    yaml_config["autoencoder"]["decoder"]["sizes"] = encoder_layer_size[::-1] #reversed for encoder
    yaml_config["autoencoder"]["nickname"] = ae_nickname
    yaml_config["data"]["nickname"] = data_nickname

    config_path = os.path.join(OUTPUT_DIR, f"config_{_latent_size}.yaml")
    with open(config_path, "w") as yaml_file:
        yaml.dump(yaml_config, yaml_file, default_flow_style=False)

    autoencoder_train_latent.main(config_path, create_latent=True,save_reco_error=True, verbose=True)
    plot_loss(WORKING_DIR,data_nickname,ae_nickname)

In [ ]:
set_random_seeds(seed=random_seed_number)


# Define the base YAML structure as a dictionary
BASE_YAML = {
    "data": {
        "source": cleaned_data_path,
        "nickname": "PLACEHOLDER",
        "id_col": "OA"
    },
    "working_dir": WORKING_DIR,
    "autoencoder": {
        "nickname": "PLACEHOLDER",
        "version": "1",
        "save_latent": "csv",
        "max_epochs": epochs,
        "batch_size": 0.01,
        "use_covariance_loss": False,
        "encoder": {
            "sizes": "PLACEHOLDER",
            "activation": "LeakyReLU",
            "sparse": {
                "topk_k": "PLACEHOLDER",
                "sparsity_loss_weight": 0.01
            }
        },
        "decoder": {
            "sizes": "PLACEHOLDER",
            "activation": "LeakyReLU"
        }
    }
}


data_nickname = "census_geodemo_sparse"

# Generate YAML files with varying latent space sizes
for encoder_layer_size in encoder_layer_sizes:
    _latent_size = encoder_layer_size[-1]  # Last element is the bottleneck size
    ae_nickname = f"bottleneck_{_latent_size}"
    yaml_config = BASE_YAML.copy()
    yaml_config["autoencoder"]["encoder"]["sizes"] = encoder_layer_size
    yaml_config["autoencoder"]["decoder"]["sizes"] = encoder_layer_size[::-1] #reversed for encoder
    yaml_config["autoencoder"]["nickname"] = ae_nickname
    yaml_config["data"]["nickname"] = data_nickname
    yaml_config["autoencoder"]["encoder"]["sparse"]["topk_k"] = int(_latent_size/2)


    config_path = os.path.join(OUTPUT_DIR, f"config_{data_nickname}_{_latent_size}.yaml")
    with open(config_path, "w") as yaml_file:
        yaml.dump(yaml_config, yaml_file, default_flow_style=False)


    autoencoder_train_latent.main(config_path, create_latent=True,save_reco_error=True, verbose=True)
    plot_loss(WORKING_DIR,data_nickname,ae_nickname)

In [ ]:
set_random_seeds(seed=random_seed_number)


# Define the base YAML structure as a dictionary
BASE_YAML = {
    "data": {
        "source": cleaned_data_path,
        "nickname": "PLACEHOLDER",
        "id_col": "OA"
    },
    "working_dir": WORKING_DIR,
    "autoencoder": {
        "nickname": "bottleneck_PLACEHOLDER",
        "version": "1",
        "save_latent": "csv",
        "max_epochs": epochs,
        "batch_size": 0.01,
        "encoder": {
            "sizes": "PLACEHOLDER",
            "activation": "LeakyReLU",
            "sparse": {
                "topk_k": "PLACEHOLDER",
                "sparsity_loss_weight": 0.01
            }
        },
        "decoder": {
            "sizes": "PLACEHOLDER",
            "activation": "LeakyReLU"
        }
    }
}


data_nickname = "census_geodemo_sparsecov"

# Generate YAML files with varying latent space sizes
for encoder_layer_size in encoder_layer_sizes:
    _latent_size = encoder_layer_size[-1]  # Last element is the bottleneck size
    ae_nickname = f"bottleneck_{_latent_size}"
    yaml_config = BASE_YAML.copy()
    yaml_config["autoencoder"]["encoder"]["sizes"] = encoder_layer_size
    yaml_config["autoencoder"]["decoder"]["sizes"] = encoder_layer_size[::-1] #reversed layer sizes than encoder
    yaml_config["autoencoder"]["nickname"] = ae_nickname
    yaml_config["data"]["nickname"] = data_nickname
    yaml_config["autoencoder"]["encoder"]["sparse"]["topk_k"] = int(_latent_size/2)


    config_path = os.path.join(OUTPUT_DIR, f"config_{data_nickname}_{_latent_size}.yaml")
    with open(config_path, "w") as yaml_file:
        yaml.dump(yaml_config, yaml_file, default_flow_style=False)


    autoencoder_train_latent.main(config_path, create_latent=True,save_reco_error=True, verbose=True)
    plot_loss(WORKING_DIR,data_nickname,ae_nickname)

In [ ]:
# Perform PCA on the scaled data
PCA_save_dir =WORKING_DIR = "../data/AE_outputs/"+data_name+"/PCA/"
os.makedirs(PCA_save_dir, exist_ok=True)
df_scaled.set_index("OA", inplace=True)


n_features = df_scaled.shape[1]
pca = PCA(n_components=n_features - 1) # Fit PCA once with the maximum number of components

transformed = pca.fit_transform(df_scaled)
for n in tqdm(bottleneck_sizes, desc="Processing PCA"):
    save_path = f"{PCA_save_dir}{n}_components.csv"

    #reconstruct the data using N components
    # Note: pca.components_[:n] gives the first n principal components
    # and pca.mean_ is the mean of the original data
    # The transformed data is projected onto the first n components and then reconstructed
    # using the mean to get the original scale back.
    # This is equivalent to the inverse transform of PCA.
    # The reconstructed data will have the same shape as the original data.
    reconstructed_pca = (transformed[:, :n] @ pca.components_[:n]) + pca.mean_
    #save the reconstructed data
    pd.DataFrame(reconstructed_pca, index=df_scaled.index, columns=df_scaled.columns).to_csv(save_path)



